## CIC-DoH-Brw-2020: data loading, `full_graph_process`, and binary trainers

Loads `l2-benign.csv` / `l2-malicious.csv` from `cic-dataset/CIRA-CIC-DoHBrw-2020`, concatenates them, runs `SrcDstGraph.full_graph_process` on the combined frame, then trains `BasicGNNClassifier` and `BasicSGNNClassifier` using `sgnn.trainers.bin_class_train`.

`BasicSGNNClassifier` uses a fixed `SrcDstGraph` inside `time_chunk_x`; when graphs differ, we train one SGNN per graph (batch size 1). `BasicGNNClassifier` is trained on both class graphs in one dataset.

Rows are subsampled by default for faster runs; set `SAMPLE_N = None` to use the full CSVs.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import torch

# Parent of this folder (`snn-stuff`) on sys.path so `import sgnn` works when cwd is `sgnn/`
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sgnn.graph_builder.graph_builder import SrcDstGraph
from sgnn.graph_builder.graph_custom_data import GraphCustomData
from sgnn.models.gnn import BasicGNNClassifier
from sgnn.models.sgnn import BasicSGNNClassifier
from sgnn.trainers.bin_class_train import (
    GraphBinaryClassificationDataset,
    train_basic_gnn_classifier_binary_nadam,
    train_basic_gnn_classifier_binary_sgd,
    train_basic_sgnn_classifier_binary_nadam,
    train_basic_sgnn_classifier_binary_sgd,
)

In [ ]:
%pip install torch snntorch pandas torch-geometric networkx


In [ ]:
DATA_DIR = ROOT / "cic-dataset" / "CIRA-CIC-DoHBrw-2020"

# Subsample for interactive runs (full malicious CSV is ~250k rows). Set to None to load everything.
SAMPLE_N = 4_000

read_kw: dict = {}
if SAMPLE_N is not None:
    read_kw["nrows"] = SAMPLE_N

df_benign = pd.read_csv(DATA_DIR / "l2-benign.csv", **read_kw)
df_malicious = pd.read_csv(DATA_DIR / "l2-malicious.csv", **read_kw)

df_benign["binary_label"] = 0
df_malicious["binary_label"] = 1

df_all = pd.concat([df_benign, df_malicious], ignore_index=True)

print(len(df_benign), "benign rows,", len(df_malicious), "malicious rows ->", len(df_all), "combined")
df_all.head()

In [ ]:
FEATURES = SrcDstGraph.DEFAULT_FEATURES

G_all = SrcDstGraph()
G_all.full_graph_process(df_all, FEATURES)

print(
    "Combined graph:",
    G_all.graph.number_of_nodes(),
    "nodes,",
    G_all.graph.number_of_edges(),
    "edges",
)

In [ ]:
# Per-class graphs for supervised binary training (one graph per class in this demo).
G_benign = SrcDstGraph()
G_benign.full_graph_process(df_benign, FEATURES)

G_malicious = SrcDstGraph()
G_malicious.full_graph_process(df_malicious, FEATURES)

# GNN: standard PyG tensors from NetworkX node attributes.
data_gnn_benign = GraphCustomData.from_networkx(
    G_benign.graph, group_node_attrs=FEATURES
)
data_gnn_malicious = GraphCustomData.from_networkx(
    G_malicious.graph, group_node_attrs=FEATURES
)

in_channels = data_gnn_benign.x.size(1)
assert in_channels == len(FEATURES)
# SGNN applies time_chunk_x to *conv outputs*: hidden dim must be divisible by len(FEATURES) (37).
HIDDEN = 74
TIMESTEPS = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 5
BATCH = 2

train_ds = GraphBinaryClassificationDataset(
    [data_gnn_benign, data_gnn_malicious],
    labels=[0, 1],
)

# --- BasicGNNClassifier: NAdam and SGD ---
gnn_nadam = BasicGNNClassifier(
    in_channels=in_channels,
    hidden_channels=HIDDEN,
    num_classes=2,
).to(device)

hist_gnn_nadam = train_basic_gnn_classifier_binary_nadam(
    gnn_nadam,
    train_ds,
    device=device,
    epochs=EPOCHS,
    batch_size=BATCH,
    lr=1e-3,
)
print("GNN NAdam final train_loss:", hist_gnn_nadam["train_loss"][-1])

gnn_sgd = BasicGNNClassifier(
    in_channels=in_channels,
    hidden_channels=HIDDEN,
    num_classes=2,
).to(device)

hist_gnn_sgd = train_basic_gnn_classifier_binary_sgd(
    gnn_sgd,
    train_ds,
    device=device,
    epochs=EPOCHS,
    batch_size=BATCH,
    lr=1e-2,
    momentum=0.9,
)
print("GNN SGD final train_loss:", hist_gnn_sgd["train_loss"][-1])

# --- BasicSGNNClassifier: one model per SrcDstGraph (required for time_chunk_x) ---
data_sgnn_benign = GraphCustomData.from_networkx_time_chunked(G_benign)
data_sgnn_malicious = GraphCustomData.from_networkx_time_chunked(G_malicious)

hist_sgnn: dict[str, dict] = {}
for name, G_src, pyg_data, label in (
    ("benign", G_benign, data_sgnn_benign, 0),
    ("malicious", G_malicious, data_sgnn_malicious, 1),
):
    ds_one = GraphBinaryClassificationDataset([pyg_data], labels=[label])

    sgnn_nadam = BasicSGNNClassifier(
        in_channels=in_channels,
        hidden_channels=HIDDEN,
        num_classes=2,
        G=G_src,
        timesteps=TIMESTEPS,
    ).to(device)
    hist_sgnn[f"{name}_nadam"] = train_basic_sgnn_classifier_binary_nadam(
        sgnn_nadam,
        ds_one,
        device=device,
        epochs=EPOCHS,
        batch_size=1,
        lr=1e-3,
    )

    sgnn_sgd = BasicSGNNClassifier(
        in_channels=in_channels,
        hidden_channels=HIDDEN,
        num_classes=2,
        G=G_src,
        timesteps=TIMESTEPS,
    ).to(device)
    hist_sgnn[f"{name}_sgd"] = train_basic_sgnn_classifier_binary_sgd(
        sgnn_sgd,
        ds_one,
        device=device,
        epochs=EPOCHS,
        batch_size=1,
        lr=1e-2,
        momentum=0.9,
    )

for k, h in hist_sgnn.items():
    print(k, "final train_loss:", h["train_loss"][-1])